# Stacks & Queues

Notes and runnable examples on the two fundamental **access-discipline** ADTs — **stack** (LIFO) and **queue** (FIFO). These aren't really data structures; they're *interfaces* you put on top of one. In Python that backing is a `list`, a `collections.deque`, or a `queue.Queue` — and the whole game is **choosing the right one**. One of the most common Python performance traps lives right here.

**Contents**

1. **What they are** — LIFO vs FIFO, an ADT over a backing
2. **Stacks** — a `list` is perfect
3. **Queues** — and the `list` trap
4. **`deque` tricks** — ring buffers & both ends
5. **The `queue` module** — thread-safe queues
6. **When to use what**
7. **Complexity cheat-sheet**

## 1. What they are

A **stack** and a **queue** are *abstract data types*: defined by their **access discipline**, not their storage.

- **Stack — LIFO** (last in, first out): `push` and `pop` act on the **same end**. Think call stack, undo history, backtracking / DFS.
- **Queue — FIFO** (first in, first out): `enqueue` at one end, `dequeue` from the **other**. Think task queue, BFS frontier, buffering.

Both are *built on top of* a concrete structure. The only real decision is picking a backing where **both** ends you need are cheap.

## 2. Stacks — a `list` is perfect

A stack only ever touches **one end**, and a Python `list` is fast exactly there: `append` and `pop()` (no index) both work at the tail in amortized O(1) — the same end the dynamic array grows from. So for a stack you usually need no wrapper at all; a bare `list` *is* the stack.

In [1]:
stack = []
stack.append("a")              # push
stack.append("b")
stack.append("c")
print("peek :", stack[-1])     # 'c'  - O(1)
print("pop  :", stack.pop())   # 'c'  - LIFO, O(1)
print("pop  :", stack.pop())   # 'b'
print("state:", stack)         # ['a']

# a thin wrapper, if you want explicit names:
class Stack:
    def __init__(self):  self._items = []
    def push(self, x):   self._items.append(x)
    def pop(self):       return self._items.pop()
    def peek(self):      return self._items[-1]
    def __len__(self):   return len(self._items)

s = Stack()
for x in (1, 2, 3): s.push(x)
print("wrapper pops:", s.pop(), s.pop(), s.pop())   # 3 2 1


peek : c
pop  : c
pop  : b
state: ['a']
wrapper pops: 3 2 1


## 3. Queues — and the `list` trap

A queue touches **both ends**: enqueue at the back, dequeue from the front. A `list` is great at the back but **terrible at the front** — `list.pop(0)` must shift *every* remaining element down one slot, so it's O(n). Drain a whole queue that way and you've written an accidental **O(n²)** loop.

`collections.deque` (a doubly linked list of blocks — see the linked-lists notebook) gives O(1) at *both* ends. **For a FIFO queue, always use `deque`, never `list.pop(0)`:**

In [2]:
import timeit
from collections import deque

print(f"{'N':>7}{'list.pop(0)':>15}{'deque.popleft':>16}{'gap':>9}")
for N in (10_000, 20_000, 40_000):
    def drain_list():
        q = list(range(N))
        while q: q.pop(0)            # O(n) each  -> O(n^2) total
    def drain_deque():
        q = deque(range(N))
        while q: q.popleft()         # O(1) each  -> O(n) total
    tl = timeit.timeit(drain_list,  number=1)
    td = timeit.timeit(drain_deque, number=1)
    print(f"{N:>7}{tl*1000:>13.2f}ms{td*1000:>14.2f}ms{tl/td:>8.0f}x")


      N    list.pop(0)   deque.popleft      gap
  10000         3.87ms          0.15ms      26x
  20000        13.70ms          0.31ms      45x
  40000        60.91ms          0.62ms      98x


The `list` time roughly **quadruples each time N doubles** — the signature of O(n²) — and its gap over `deque` widens accordingly, while `deque` draining stays linear. This is the single most common "why is my Python slow" bug in queue code.

## 4. `deque` tricks — ring buffers & both ends

Because `deque` owns both ends, it does a few things a `list` can't do cheaply:

- **`maxlen`** turns it into a fixed-size **ring buffer**: once full, each `append` drops the opposite end. Ideal for "last N" sliding windows, rolling logs, bounded history.
- **`appendleft` / `popleft`** — O(1) front operations (the whole point).
- **`rotate(k)`** — cycle the ends in O(k).

In [3]:
from collections import deque

window = deque(maxlen=3)        # keep only the 3 most recent items
for x in [1, 2, 3, 4, 5]:
    window.append(x)
    print(f"  append {x} -> {list(window)}")

dq = deque([1, 2, 3, 4, 5])
dq.rotate(2)                    # last 2 wrap around to the front
print("rotate(2):", dq)


  append 1 -> [1]
  append 2 -> [1, 2]
  append 3 -> [1, 2, 3]
  append 4 -> [2, 3, 4]
  append 5 -> [3, 4, 5]
rotate(2): deque([4, 5, 1, 2, 3])


## 5. The `queue` module — thread-safe queues

`collections.deque` is the right call for a **single-threaded** stack or queue. When you pass work **between threads** (producer/consumer), reach for the `queue` module instead. Its `Queue` (FIFO), `LifoQueue` (stack), and `PriorityQueue` wrap a deque/heap with **locking** and **blocking** semantics: `get()` waits for an item, `put()` can wait for free space, and `task_done()` / `join()` let a producer block until all work is processed.

The trade-off is real: that synchronization adds overhead, so **don't** use `queue.Queue` for plain single-threaded buffering — use a `deque`.

In [4]:
from queue import Queue, LifoQueue, PriorityQueue

q = Queue()
for x in (1, 2, 3): q.put(x)
print("Queue         (FIFO):", [q.get() for _ in range(3)])    # 1 2 3

lifo = LifoQueue()
for x in (1, 2, 3): lifo.put(x)
print("LifoQueue     (LIFO):", [lifo.get() for _ in range(3)])  # 3 2 1

pq = PriorityQueue()
for x in (3, 1, 2): pq.put(x)
print("PriorityQueue (min) :", [pq.get() for _ in range(3)])    # 1 2 3


Queue         (FIFO): [1, 2, 3]
LifoQueue     (LIFO): [3, 2, 1]
PriorityQueue (min) : [1, 2, 3]


## 6. When to use what

| You need... | Reach for | Why |
|---|---|---|
| A stack (LIFO), single-threaded | `list` | `append`/`pop()` at the tail are O(1); no wrapper needed |
| A queue (FIFO), single-threaded | `collections.deque` | O(1) at both ends; never `list.pop(0)` |
| A bounded "last N" / ring buffer | `deque(maxlen=N)` | auto-drops the far end on overflow |
| A priority queue | `heapq` (or `queue.PriorityQueue` for threads) | O(log n) by priority |
| To hand work between threads | `queue.Queue` / `LifoQueue` / `PriorityQueue` | locking + blocking `get`/`put`, `task_done`/`join` |
| A queue across processes | `multiprocessing.Queue` | picklable, IPC-safe |

## 7. Complexity cheat-sheet

| Operation | `list` (as stack) | `list` (as queue) | `deque` | `queue.Queue` |
|---|:---:|:---:|:---:|:---:|
| Push / enqueue | O(1) append† | O(1) append† | O(1) both ends | O(1)‡ |
| Pop / dequeue | O(1) `pop()` | **O(n)** `pop(0)` ✗ | O(1) both ends | O(1)‡ |
| Peek | O(1) | O(1) | O(1) at ends | — (`get` removes) |
| Random index | O(1) | O(1) | O(n) | — |
| Thread-safe? | no | no | per-op only§ | yes (by design) |

<sub>† amortized (dynamic-array growth). &middot; ‡ plus lock overhead. &middot; § a `deque`'s own `append`/`appendleft`/`pop`/`popleft` are each atomic under the GIL, but it isn't a coordination primitive — use `queue.Queue` for producer/consumer.</sub>